# Root-TransUNet Inference Demonstration

This notebook demonstrates how to load the Root-TransUNet model to perform segmentation on *Arabidopsis* root images.

In [1]:
import os
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Assuming your model is defined in the 'networks' folder
from networks.vit_seg_modeling import VisionTransformer as ViT_seg
from networks.vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

# --- Configuration Parameters ---
class Args:
    num_classes = 2
    img_size = 512
    vit_name = "R50-ViT-B_16"
    vit_patches = 16
    n_skip = 3
    # Use relative paths! Recommend placing weights in a 'model_weights' folder
    model_path = "./model/best_model.pth" 
    input_dir = "./test_samples"
    save_dir = "./output/predictions"

args = Args()

# Create output directory if it doesn't exist
os.makedirs(args.save_dir, exist_ok=True)

## 1. Load Pre-trained Model

Initialize the Root-TransUNet architecture and load the weights downloaded from the GitHub Release.

In [4]:
def load_model(weights_path):
    # Adjust this based on your actual initialization code
    config_vit = CONFIGS_ViT_seg[args.vit_name]
    net = ViT_seg(config_vit, img_size=args.img_size, num_classes=args.num_classes).cuda()
    net.load_state_dict(torch.load(weights_path))
    net.eval()
    print(f"Successfully loaded model from {weights_path}")
    return None # return net

# net = load_model(args.model_path)

## 2. Interactive Prediction & Visualisation

Visualize the segmentation performance on a single sample.

In [3]:
def visualize_prediction(img_path, model):
    # 1. Load original image
    raw_img = Image.open(img_path).convert('RGB')
    
    # 2. Call your inference function (assuming it's named infer_single_rgb)
    # pred_mask = infer_single_rgb(img_path, model, args.img_size)
    
    # Simulating a mask for demonstration purposes
    pred_mask = np.random.randint(0, 2, (args.img_size, args.img_size)) 

    # 3. Plot comparison
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(raw_img)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Root-TransUNet Prediction")
    plt.imshow(pred_mask, cmap='gray')
    plt.axis('off')
    plt.show()

# Test on the first image in the input directory
# test_imgs = os.listdir(args.input_dir)
# if test_imgs:
#     visualize_prediction(os.path.join(args.input_dir, test_imgs[0]), net)

## 3. Batch Prediction

Iterate through the input folder, predict masks, and save them to the output directory.

In [5]:
def save_mask(mask_array, save_path):
    # mask_array is expected to be boolean or 0/1 integers
    mask_to_save = (mask_array * 255).astype(np.uint8)
    img = Image.fromarray(mask_to_save)
    img.save(save_path)

# Your batch inference loop...
# for img_name in os.listdir(args.input_dir):
#     IMG_PATH = os.path.join(args.input_dir, img_name)
#     pred_512 = infer_single_rgb(IMG_PATH, net, args.img_size, n_classes=args.num_classes, tta=True)
#     save_path = os.path.join(args.save_dir, img_name)
#     save_mask(pred_512, save_path)